# Intro to Hugging Face
In this exercise, we will explore the capabilities of transformer-based models for natural language processing using the Hugging Face `transformers` library. We will use the `sentence-transformers` package to extract features from text data and the `transformers` library to perform sentiment analysis and text generation tasks.

By the end of this exercise, you will have learned the basics of how to:
- Extract features from text data using transformer-based models
- Generate text using transformer-based models
- Perform sentiment analysis on text data


## Using Notebook Environments 
1. To run a cell, press `shift + enter`. The notebook will execute the code in the cell and move to the next cell. If the cell contains a markdown cell (text only), it will render the markdown and move to the next cell.
2. Since cells can be executed in any order and variables can be over-written, you may at some point feel that you have lost track of the state of your notebook. If this is the case, you can always restart the kernel by clicking `Runtime` in the menu bar (if you're using Colab) and selecting `Restart runtime`. This will clear all variables and outputs.
3. The final variable in a cell will be printed on the screen. If you want to print multiple variables, use the `print()` function as usual.

Notebook environments support code cells and markdown (text) cells. For the purposes of this workshop, markdown cells are used to provide high-level explanations of the code. More specific details are provided in the code cells themselves in the form of comments (lines beginning with `#`).

**NOTE: Please only complete the BONUS TASKS at the end if you have finished everything else**.

## Environment Setup

In [ ]:
import sys
if 'google.colab' in sys.modules:  # If in Google Colab environment
    
    # Installing requisite packages
    !pip install transformers sentence-transformers &> /dev/null

    # Import user data for secret token access
    from google.colab import userdata

We begin by loading the requisite packages. For those coming from R, packages in Python are sometimes given shorter names for use in the code via the `import <name> as <nickname>` syntax (e.g. `import pandas as pd`). These are usually standardized nicknames. We here make use of:

1. `pandas`: A popular package for reading and manipulating data in python.
2. `sentence_transformers`: A package for extracting features from text data using transformer-based models.
3. `transformers`: A Hugging Face package for loading and manipulating transformer-based models.
4. `huggingface_hub`: Allows you to interact with the Hugging Face Hub (e.g., via the `InferenceClient`)

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from huggingface_hub import InferenceClient

## Feature Extraction

The following code begins by extracting features (or embeddings) from the text data, which are numerical representations of the meaning of text, using the `sentence-transformers` package. To start, it uses three sentences that are placed in a list of strings. This list is provided as input to the model.

The code makes use of the `all-MiniLM-L6-v2` model, which is a small and efficient embedding model, to extract features from the sentences. The model will encode the sentences into 384-dimensional vector representations. The cell will then print the features as a pandas dataframe for easy viewing. 

Run the cell below. 

In [ ]:
# Define sentences
sentences = [
    "I feel great this morning",
    "I am feeling very good today",
    "I am feeling terrible"
]

# Load the pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Extract features
features = model.encode(sentences)

# Print the features as a pandas dataframe
pd.DataFrame(features, index=sentences)

**TASK 1**: Have a scroll through the features printed by the cell. Can you see that the features of the first two sentences are more similar to each other (i.e., have similar numerical values) than they are to the third sentence? Why do you think this is the case?

**TASK 2**: Try adding another sentence to the `sentences` list defined above by copying one of the existing sentences but replacing one or two words with a synonym. For instance, you could change "I feel *great* this morning" to "I feel *fantastic* this morning". Then rerun the cell. What do you notice about the features of this altered sentence compared to the original?

**BONUS TASK**: Try replacing `'all-MiniLM-L6-v2'` with another `sentence-transformers`-compatible model. You can find other compatible models [here](https://www.sbert.net/docs/sentence_transformer/pretrained_models.html#) (under 'Original Models'), along with details about their sizes and performances.

## Text Generation (local)
This section uses the `transformers` text generation pipeline. The cell below begins by loading the pipeline with the `gpt2` model--the smaller and open great-grandparent of ChatGPT. **We use this model for introductory purposes only: GPT-2 is not used for serious applications nowadays, and we would not recommend doing so (you will soon see why).** It is nevertheless fun to play around with, and insightful to get an impression for how far language models have come in the last several years.

The cell then defines a prompt. The prompt is a starting point that the model uses to generate text. GPT-2 will use it to generate text that is likely to follow the prompt. We set the `max_new_tokens` parameter to 100 to limit the length of the generated text to 100 tokens, and set 'return_full_text'=False' so the output only contains the text generated by the model (i.e., excluding the prompt).

Run the cell below.

In [ ]:
pipe = pipeline('text-generation', model='openai-community/gpt2')

prompt = """
Once upon a time in a land far far away, there was a young prince named John.
He was known for his bravery and courage. One day, he decided to go on an adventure to explore the unknown lands.
"""

# Generate text based on the prompt
output = pipe(prompt, max_new_tokens=100, return_full_text=False)

# Print the generated text
print(output[0]['generated_text'])

**TASK 3**: Try changing the text in the variable `prompt` to a specific question that you wish the model to answer (e.g., `"What is your name?"`). Feel free to play around with the `max_new_tokens` parameter as well to prevent the output getting curtailed too early.

**TASK 4**: Try replacing `'openai-community/gpt2'` above with [`'Qwen/Qwen3-0.6B'`](https://huggingface.co/Qwen/Qwen3-0.6B) to see how the quality and style of the generated text improves (especially when responding to questions). Note that `'Qwen/Qwen3-0.6B'` is still small by LLM standards (we will try larger models later when we start using the GPU instead of the CPU, which we are currently using).

# Text Generation (Hugging Face API)
Some open-weight models are too large for us to run locally. In such cases, we can instead run models from remote servers via an API. The code below demonstrates how to run a 27B parameter model, [`"google/gemma-3-27b-it"`](https://huggingface.co/google/gemma-3-27b-it), using the Hugging Face `InferenceClient`. In order to run this, you will need to [create a HuggingFace account](https://huggingface.co/join). **You will also need to generate a Hugging Face access token** by navigating to [your Hugging Face profile settings](https://huggingface.co/settings/tokens), clicking `+Create new token`, setting `Token type` to `Read`, and giving it a name (e.g., 'HF'). You can then copy-paste your token into your Colab environment by clicking the `secrets` symbol on the left-hand sidebar, clicking `+ Add new secret`, and then pasting the token in the `Value` box and giving it the name `'HF'`. Finally, you will (unfortunately) have to [add a couple of dolars of credit](https://huggingface.co/settings/billing) to your account, since Hugging Face accounts no longer receive monthly free inference credits. 

In [ ]:
hf_token=userdata.get('HF')

client = InferenceClient(token=hf_token)
output = client.chat_completion(
    model="google/gemma-3-27b-it",
    max_tokens=100,
    messages=[
        {"role": "user", "content": prompt}
        ]
)
print(output.choices[0].message.content)

## Sentiment Analysis 
In addition to feature extraction, Hugging Face's `transformers` library provides a high-level API for a variety of other tasks. These tasks can be viewed in the left-hand panel under `Tasks > Natural Language Processing` on the [Hugging Face model hub](https://huggingface.co/models). Many of these tasks include models that have been fine-tuned on specific datasets to perform well on the task at hand. As an example of such a task, we will now use the `'text-classification'` pipeline to do sentiment classification on a few sentences.

The cell below will now load the `transformers` `'text-classification'` pipeline with [`'tabularisai/multilingual-sentiment-analysis'`](https://huggingface.co/tabularisai/multilingual-sentiment-analysis) to predict the sentiment of sentences for the same sentences as before.

Run the cell below.

In [ ]:
# Define the sentences
sentences = [
    "I feel great this morning",
    "I am feeling very good today",
    "I am feeling terrible"
]

# Load sentiment analysis pipeline
pipe = pipeline('text-classification', model='tabularisai/multilingual-sentiment-analysis')

# Predict sentiment of the sentences
sentiments = pipe(sentences)

# Print the predicted sentiments as a pandas dataframe
pd.DataFrame(sentiments, index=sentences)

As you can see, not only does the model predict the sentiment of the sentences (`'label'`), but it also provides a confidence score for each prediction (`'score'`).

**TASK 5:** Try checking out the languages supported by the model on the [model card](https://huggingface.co/tabularisai/multilingual-sentiment-analysis) (under 'Model Details'). Use a translation software of your choice to translate the above sentences into another supported langauge. Do the sentiment labels remain roughly the same?